# Data Cleaning and Handling
## Dataset: Merged Dataset

**Author:** Oknardo Tulung  
**LinkedIn:** https://www.linkedin.com/in/oknardo-tulung/  
**GitHub:** https://github.com/oknardo/Home_Credit_Scorecard_Model

---

## 📌 Project Overview
This notebook performs comprehensive data cleaning and feature engineering on the merged dataset produced from `MERGED_DATASET.ipynb`. The goal is to prepare a clean, model-ready dataset by handling missing values, treating outliers, engineering new features, and encoding categorical variables.

---

## 🎯 Objectives
- Drop irrelevant, redundant, and high-missing-rate features
- Handle missing values across all feature groups with appropriate strategies
- Treat outliers in numerical features using capping
- Engineer new features from existing ones to improve predictive signal
- Encode categorical features for modeling
- Produce a final clean dataset ready for scorecard modeling

---

## 🔍 Analysis Scope
The approach includes:
- Feature dropping based on missingness, variance, and redundancy
- Missing value imputation per feature group
- Outlier capping at 1st and 99th percentiles for skewed features
- Special handling for anomalous encodings (e.g. DAYS_EMPLOYED = 365,243)
- Derived feature creation from domain knowledge
- Label encoding and WoE encoding for categorical features

---

## 🛠 Tools & Libraries
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn

---

## 📊 Output
The output of this notebook will serve as:
- A clean, model-ready dataset for scorecard development
- Input for LightGBM, XGBoost, and Logistic Regression models
- A documented and reproducible data preparation pipeline

# Importing Library

In [23]:
# Installation Library
!pip install seaborn


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
# Install scikit-learn
!pip install scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.1 MB 2.8 MB/s eta 0:00:03
   --------- ------------------------------ 1.8/8.1 MB 3.2 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/8.1 MB 3.1 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.1 MB 3.1 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.1 MB 2.8 MB/s eta 0:00:02
   ------------------ --------------------- 3.7/8.1 MB 2.7 MB/s eta 0:00:02
   ---------------------- ----------------- 4.5/8.1 MB 2.8 MB/s eta 0:00:02
   ------------------------- -------------- 5.2/8.1 MB 2.9 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.1 MB 2.9 MB/s eta 0:00:01
   -------------------------------- ------- 6.6/8.1 MB 2.9 MB/s eta 0:00:01
   ------------------------------------ --- 7.3/8.1 MB 3.0 MB/s eta 0:00:01
   -----------------------


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
# Hide Warning
import warnings
warnings.filterwarnings('ignore')

# Importing Library
import pandas as pd
# Setting Pandas Row Display Max
pd.set_option('display.max_rows', None)

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder


# Importing Dataset

In [3]:
# Load merged dataset
merged_path = r'D:\Python\Projects\Project Scorecard Model Home Credit Indonesia\home-credit-default-risk\merged_dataset.csv'
df = pd.read_csv(merged_path)
print(f"Shape: {df.shape}")

Shape: (307511, 265)


# 1. Drop Irrelevant Features

This section removes features that are unlikely to contribute meaningful signal to the model. Dropping irrelevant features reduces noise, speeds up training, and improves model interpretability.

The approach includes:
- **Near-constant features**: features with extremely low variance across all applicants
- **Highly redundant features**: features confirmed as perfectly or near-perfectly correlated in EDA
- **High missing rate features**: features with missing rates above the defined threshold where signal is insufficient to justify retention

## 1.1 Near-Constant Features

In [4]:
# Drop near-constant features identified in EDA_application_train
near_constant_cols = [
    'FLAG_MOBIL',
    'FLAG_CONT_MOBILE',
    'HOUSETYPE_MODE',
    'EMERGENCYSTATE_MODE',
]

df.drop(columns=near_constant_cols, inplace=True)
print(f"Dropped {len(near_constant_cols)} near-constant features.")
print(f"Shape after drop: {df.shape}")

Dropped 4 near-constant features.
Shape after drop: (307511, 261)


## 1.2 Highly Redundant Features

In [5]:
# Drop redundant features confirmed in EDA notebooks
redundant_cols = [
    # application_train: near-perfect correlation with AMT_CREDIT
    'AMT_GOODS_PRICE',

    # application_train: near-perfect correlation with CNT_CHILDREN
    'CNT_FAM_MEMBERS',

    # application_train: building features - retain only _AVG, drop _MODE and _MEDI
    'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE',
    'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE',
    'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE',
    'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MODE',
    'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
    'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI',
    'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI',
    'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'APARTMENTS_MEDI',

    # credit_card_balance: perfectly correlated with AMT_BALANCE
    'AMT_RECEIVABLE_PRINCIPAL',
    'AMT_TOTAL_RECEIVABLE',
]

df.drop(columns=[col for col in redundant_cols if col in df.columns], inplace=True)
print(f"Dropped {len(redundant_cols)} redundant features.")
print(f"Shape after drop: {df.shape}")

Dropped 32 redundant features.
Shape after drop: (307511, 231)


## 1.3 High Missing Rate Features

In [6]:
# Drop features with missing rate above 80%
missing_rate = df.isnull().sum() / len(df) * 100
high_missing_cols = missing_rate[missing_rate > 80].index.tolist()

print(f"Features with missing rate > 80%: {len(high_missing_cols)}")
print(high_missing_cols)

df.drop(columns=high_missing_cols, inplace=True)
print(f"\nShape after drop: {df.shape}")

Features with missing rate > 80%: 1
['CC_AMT_DRAWINGS_ATM_CURRENT_MEAN']

Shape after drop: (307511, 230)


### Summary

Feature dropping reduced the dataset from **265 to 230 features**, removing **35 features** in total across three categories.

- **Near-constant features dropped (4)**: `FLAG_MOBIL`, `FLAG_CONT_MOBILE`, `HOUSETYPE_MODE`, `EMERGENCYSTATE_MODE` — all confirmed with near-zero variance in `EDA_application_train.ipynb`
- **Redundant features dropped (32)**: building `_MODE` and `_MEDI` variants, `AMT_GOODS_PRICE`, `CNT_FAM_MEMBERS`, and perfectly correlated credit card balance features
- **High missing rate features dropped (1)**: `CC_AMT_DRAWINGS_ATM_CURRENT_MEAN` (80.12% missing), the only feature exceeding the 80% threshold

The remaining **230 features** retain all meaningful signal identified across EDA notebooks while reducing noise and dimensionality before imputation and modeling.

# 2. Missing Value Handling

This section handles missing values across all feature groups using strategies defined in individual EDA notebooks. Each feature group is treated separately based on the nature and cause of its missingness.

The approach includes:
- **Impute with 0**: for count, proportion, binary flag, and amount features where missing indicates absence
- **Impute with median**: for numerical features where missing is random or tied to specific subgroups
- **Impute with mode**: for categorical features with low missing rates
- **Create missing indicators**: for features where missingness carries predictive signal

## 2.1 Application Train Features

In [8]:
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']:
    df[col] = df[col].fillna(df[col].median())

df['OWN_CAR_AGE'] = df['OWN_CAR_AGE'].fillna(0)
df['AMT_ANNUITY'] = df['AMT_ANNUITY'].fillna(df['AMT_ANNUITY'].median())

for col in [c for c in df.columns if c.startswith('AMT_REQ_CREDIT_BUREAU')]:
    df[col] = df[col].fillna(df[col].median())

for col in ['NAME_TYPE_SUITE', 'OCCUPATION_TYPE', 'FONDKAPREMONT_MODE', 'WALLSMATERIAL_MODE']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

for col in [c for c in df.columns if c.endswith('_AVG')]:
    df[col] = df[col].fillna(df[col].median())

df['TOTALAREA_MODE'] = df['TOTALAREA_MODE'].fillna(df['TOTALAREA_MODE'].median())

for col in ['OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
            'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
            'DAYS_LAST_PHONE_CHANGE']:
    df[col] = df[col].fillna(df[col].median())

## 2.2 Bureau Features

In [9]:
for col in [c for c in df.columns if c.startswith('BUREAU_')]:
    df[col] = df[col].fillna(0)

## 2.3 Previous Application Features

In [10]:
for col in [c for c in df.columns if c.startswith('PREV_')]:
    df[col] = df[col].fillna(0)

## 2.4 POS_CASH_balance Features

In [11]:
for col in [c for c in df.columns if c.startswith('POS_')]:
    df[col] = df[col].fillna(0)

## 2.5 Credit Card Balance Features

In [12]:
for col in [c for c in df.columns if c.startswith('CC_')]:
    df[col] = df[col].fillna(0)

## 2.6 Installments Payments Features

In [13]:
df['INST_AMT_PAYMENT_RATIO'] = df['INST_AMT_PAYMENT_RATIO'].fillna(1)
for col in [c for c in df.columns if c.startswith('INST_')]:
    df[col] = df[col].fillna(0)

## Verified


In [14]:
# Verify
total_missing = df.isnull().sum().sum()
print(f"Total remaining missing values: {total_missing}")
print(f"Shape: {df.shape}")

Total remaining missing values: 0
Shape: (307511, 230)


# 3. Outlier Treatment

This section handles outliers in numerical features using capping and special encoding strategies identified during EDA. The goal is to reduce the influence of extreme values without losing the underlying signal.

The approach includes:
- **Capping at 1st and 99th percentile** for skewed numerical features with extreme outliers
- **Special handling** for anomalous encodings identified in EDA

## 3.1 Special Handling

In [16]:
# DAYS_EMPLOYED: replace 365243 (encoding anomaly) with NaN then impute with median
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].fillna(df['DAYS_EMPLOYED'].median())

# CODE_GENDER: replace XNA with mode
df['CODE_GENDER'] = df['CODE_GENDER'].replace('XNA', df['CODE_GENDER'].mode()[0])

# NAME_FAMILY_STATUS: replace Unknown with mode
df['NAME_FAMILY_STATUS'] = df['NAME_FAMILY_STATUS'].replace('Unknown', df['NAME_FAMILY_STATUS'].mode()[0])

# NAME_INCOME_TYPE: group rare categories
rare_income = ['Maternity leave', 'Student', 'Businessman']
df['NAME_INCOME_TYPE'] = df['NAME_INCOME_TYPE'].replace(rare_income, 'Other')

# AMT_CREDIT_SUM_DEBT and AMT_CREDIT_SUM_LIMIT: clip negative values to 0
# (handled via capping in 3.2)

print("Special handling completed.")
print(f"DAYS_EMPLOYED anomaly check: {(df['DAYS_EMPLOYED'] == 365243).sum()} remaining")
print(f"CODE_GENDER XNA check: {(df['CODE_GENDER'] == 'XNA').sum()} remaining")

Special handling completed.
DAYS_EMPLOYED anomaly check: 0 remaining
CODE_GENDER XNA check: 0 remaining


## 3.2 Capping Strategy

In [17]:
# Define features to cap at 1st and 99th percentile
cap_cols = [
    # application_train
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
    'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',

    # bureau
    'BUREAU_AMT_CREDIT_SUM_MEAN', 'BUREAU_AMT_CREDIT_SUM_SUM',
    'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', 'BUREAU_AMT_CREDIT_SUM_DEBT_SUM',
    'BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN', 'BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX',
    'BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN', 'BUREAU_AMT_ANNUITY_MEAN',

    # previous_application
    'PREV_AMT_CREDIT_MEAN', 'PREV_AMT_CREDIT_SUM',
    'PREV_AMT_APPLICATION_MEAN', 'PREV_AMT_APPLICATION_SUM',
    'PREV_AMT_GOODS_PRICE_MEAN',

    # POS_CASH_balance
    'POS_SK_DPD_MEAN', 'POS_SK_DPD_MAX',
    'POS_SK_DPD_DEF_MEAN', 'POS_SK_DPD_DEF_MAX',

    # credit_card_balance
    'CC_AMT_BALANCE_MEAN', 'CC_AMT_BALANCE_MAX',
    'CC_SK_DPD_MEAN', 'CC_SK_DPD_MAX',
    'CC_SK_DPD_DEF_MEAN', 'CC_SK_DPD_DEF_MAX',
    'CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN',
    'CC_UTILIZATION_RATIO',

    # installments_payments
    'INST_AMT_INSTALMENT_MEAN', 'INST_AMT_PAYMENT_MEAN',
    'INST_DAYS_PAYMENT_DIFF_MAX', 'INST_AMT_PAYMENT_DIFF_MEAN',
    'INST_AMT_PAYMENT_DIFF_MAX', 'INST_AMT_PAYMENT_RATIO',
]

# Apply capping
for col in cap_cols:
    if col in df.columns:
        p01 = df[col].quantile(0.01)
        p99 = df[col].quantile(0.99)
        df[col] = df[col].clip(lower=p01, upper=p99)

print(f"Capping applied to {len([c for c in cap_cols if c in df.columns])} features.")
print(f"Shape: {df.shape}")

Capping applied to 38 features.
Shape: (307511, 230)


# 4. Feature Engineering

This section creates new features from existing ones based on domain knowledge and insights from EDA notebooks. The goal is to derive additional predictive signal before modeling.

The approach includes:
- **Derived numerical features** from ratios, differences, and combinations of existing features
- **Encoding categorical features** for use in machine learning models

## 4.1 Derived Features

In [18]:
# Credit to income ratio
df['CREDIT_TO_INCOME_RATIO'] = (df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']).round(4)

# Annuity to income ratio
df['ANNUITY_TO_INCOME_RATIO'] = (df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']).round(4)

# Credit to annuity ratio (loan term proxy)
df['CREDIT_TO_ANNUITY_RATIO'] = (df['AMT_CREDIT'] / df['AMT_ANNUITY']).round(4)

# Age in years (absolute value of DAYS_BIRTH)
df['AGE_YEARS'] = (df['DAYS_BIRTH'] / -365).round(2)

# Employment length in years
df['EMPLOYED_YEARS'] = (df['DAYS_EMPLOYED'] / -365).round(2)

# Employment to age ratio
df['EMPLOYED_TO_AGE_RATIO'] = (df['EMPLOYED_YEARS'] / df['AGE_YEARS']).round(4)

# EXT_SOURCE combinations
df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1).round(4)
df['EXT_SOURCE_MIN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].min(axis=1).round(4)
df['EXT_SOURCE_PROD'] = (df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']).round(4)

# Family members per child
df['INCOME_PER_PERSON'] = (df['AMT_INCOME_TOTAL'] / df['CNT_CHILDREN'].replace(0, 1)).round(4)

# Bureau delinquency cross signal
df['BUREAU_INST_DELINQUENCY'] = (
    df['BUREAU_BB_PROP_DELINQUENT_MEAN'] + df['INST_PROP_LATE']
).round(4)

print(f"Derived features created.")
print(f"Shape after feature engineering: {df.shape}")

Derived features created.
Shape after feature engineering: (307511, 241)


## 4.2 Encoding Categorical Features

In [28]:
# Check remaining categorical features
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical features to encode: {len(cat_cols)}")
print(cat_cols)

Categorical features to encode: 14
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'WALLSMATERIAL_MODE']


In [29]:
# Label encoding for all categorical features
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print(f"\nLabel encoding applied to {len(cat_cols)} features.")
print(f"Shape after encoding: {df.shape}")

# Verify no object columns remain
remaining_cat = df.select_dtypes(include='object').columns.tolist()
print(f"Remaining categorical features: {len(remaining_cat)}")


Label encoding applied to 14 features.
Shape after encoding: (307511, 241)
Remaining categorical features: 0


# 5. Final Dataset

This section provides a final overview of the clean dataset and saves it for use in modeling.

The approach includes:
- **Reviewing** the final shape, feature composition, and data types
- **Verifying** no missing values or object columns remain
- **Saving** the clean dataset for modeling

## 5.1 Final Shape & Overview

In [30]:
# Final shape
print(f"Final dataset shape: {df.shape}")
print(f"Total features: {df.shape[1]}")
print(f"Total applicants: {df.shape[0]}")

# Data types summary
print(f"\nData types:")
print(df.dtypes.value_counts())

# Missing values verification
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# Target distribution
print(f"\nTarget distribution:")
print(df['TARGET'].value_counts())
print(f"Default rate: {df['TARGET'].mean() * 100:.2f}%")

Final dataset shape: (307511, 241)
Total features: 241
Total applicants: 307511

Data types:
float64    189
int64       52
Name: count, dtype: int64

Total missing values: 0

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64
Default rate: 8.07%


## 5.2 Save Clean Dataset

In [31]:
# Save clean dataset
clean_output_path = r'D:\Python\Projects\Project Scorecard Model Home Credit Indonesia\home-credit-default-risk\clean_dataset.csv'
df.to_csv(clean_output_path, index=False)
print(f"Clean dataset saved successfully!")
print(f"Path: {clean_output_path}")
print(f"Shape: {df.shape}")

Clean dataset saved successfully!
Path: D:\Python\Projects\Project Scorecard Model Home Credit Indonesia\home-credit-default-risk\clean_dataset.csv
Shape: (307511, 241)
